# tocandira, round 1 of 2: generate labeled positions

Builds the engine, plays self play games at a fixed depth and records, for every position,
the search score, nokamute's static evaluation and the final outcome. Positions are encoded
into NNUE style features and saved to Drive as one shard per 1000 games.

One notebook file per parallel run: save a copy in Drive per instance (two tabs of the same
notebook would share one runtime), set `INSTANCE` to 0 in one copy and 1 in the other, run
both to completion. Instances take disjoint shards, so they never collide.
Shards already on Drive are skipped, so a disconnected session just gets re-run.

CPU bound, the GPU is idle: pick the runtime with the most CPUs. TPU v6e-1 has 44 vCPUs
(about 4.6 units/hour), L4 has 12 (about 1.7 units/hour); the first cell prints the count.
30 shards cost about 4 units either way.

In [ ]:
INSTANCE, N_INSTANCES = 0, 2   # this copy does shards with shard % N_INSTANCES == INSTANCE
SHARDS = 30                     # 1000 games each, about 75k positions per shard
GAMES_PER_SHARD = 1000
DEPTH = 4
RANDOM_MOVE_PROB = 0.1
DRIVE_DIR = "/content/drive/MyDrive/tocandira/data/d4"

import os
print("cpus:", os.cpu_count())

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
os.makedirs(DRIVE_DIR, exist_ok=True)

In [ ]:
%%bash
set -e
if ! command -v cargo >/dev/null 2>&1; then curl -sSf https://sh.rustup.rs | sh -s -- -y -q >/dev/null; fi
source "$HOME/.cargo/env"
cd /content
# Cloned to /content/repo, not /content/tocandira: a folder named like the package would shadow it on sys.path.
if [ -d repo ]; then (cd repo && git pull -q && git submodule update --init -q); else git clone -q --recurse-submodules https://github.com/abstratovcm/tocandira repo; fi
cd /content/repo/engine/nokamute
CARGO_PROFILE_RELEASE_LTO=true CARGO_PROFILE_RELEASE_CODEGEN_UNITS=1 cargo build --release -q
cd /content/repo && pip install -q -e .
./engine/nokamute/target/release/nokamute --help | head -1

In [ ]:
import gzip
import shutil
import subprocess
import time

WORKERS = os.cpu_count() or 2
for shard in range(INSTANCE, SHARDS, N_INSTANCES):
    npz = f"{DRIVE_DIR}/shard-{shard:03d}.npz"
    if os.path.exists(npz):
        print("exists, skipping", npz)
        continue
    jsonl = f"/content/shard-{shard:03d}.jsonl"
    t0 = time.time()
    subprocess.run(
        ["hive-label", "--games", str(GAMES_PER_SHARD), "--depth", str(DEPTH),
         "--random-move-prob", str(RANDOM_MOVE_PROB), "--seed", str(1000 + shard),
         "--workers", str(WORKERS), "--out", jsonl],
        check=True,
    )
    subprocess.run(["hive-encode", jsonl, "--out", "/content/shard.npz", "--workers", str(WORKERS)], check=True)
    with open(jsonl, "rb") as src, gzip.open(f"{DRIVE_DIR}/shard-{shard:03d}.jsonl.gz", "wb") as dst:
        shutil.copyfileobj(src, dst)  # the move lists, in case the encoding changes later
    shutil.move("/content/shard.npz", npz)
    os.remove(jsonl)
    print(f"shard {shard}: {time.time() - t0:.0f}s -> {npz}")

Reference: 7 workers on a Ryzen 7 5700G label about 220 positions/s at depth 4, so a 1000
game shard takes about 6 minutes there; scale by the CPU count printed above. Each shard is
about 1.5 MB of npz plus a few MB of gzipped move lists.